# 1. Data Understanding

## 1.1 Campaigns

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
Path.cwd()

WindowsPath('C:/Users/marta/OneDrive/Desktop/PROJEKTY/retail-promotion-effectiveness/notebooks')

In [4]:
campaign_desc = pd.read_csv("../data/raw/campaign_desc.csv")

In [5]:
campaign_desc.head()

,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684


In [6]:
campaign_desc.shape

(30, 4)

In [7]:
campaign_desc.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   DESCRIPTION  30 non-null     str  
 1   CAMPAIGN     30 non-null     int64
 2   START_DAY    30 non-null     int64
 3   END_DAY      30 non-null     int64
dtypes: int64(3), str(1)
memory usage: 1.1 KB


In [8]:
campaign_desc["CAMPAIGN"].is_unique

True

In [9]:
campaign_desc["DESCRIPTION"].value_counts()

DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64

In [10]:
campaign_desc[["START_DAY", "END_DAY"]].agg(["min", "max"])

,START_DAY,END_DAY
min,224,264
max,659,719


In [11]:
# Calculate campaign duration (inclusive of start and end days)
campaign_desc["DURATION_DAYS"] = campaign_desc["END_DAY"] - campaign_desc["START_DAY"] + 1

In [12]:
# Summarize campaign duration by campaign type
campaign_desc.groupby("DESCRIPTION")["DURATION_DAYS"].agg(["count", "min", "median", "max"])

,count,min,median,max
DESCRIPTION,,,,
TypeA,5,41,48.0,56
TypeB,19,33,33.0,62
TypeC,6,33,65.0,162


In [13]:
# Check for campaigns with zero or negative duration
(campaign_desc["DURATION_DAYS"] <= 0).sum()

np.int64(0)

In [14]:
# Identify campaign pairs with overlapping periods
pairs = campaign_desc.merge(campaign_desc, how="cross", suffixes=("_1", "_2"))

overlaps = pairs[
    (pairs["CAMPAIGN_1"] < pairs["CAMPAIGN_2"])
    & (pairs["START_DAY_1"] <= pairs["END_DAY_2"])
    & (pairs["START_DAY_2"] <= pairs["END_DAY_1"])
]

len(overlaps)

74

In [15]:
# Calculate the number of overlapping days for each campaign pair
overlaps = overlaps.copy()

overlaps["OVERLAP_DAYS"] = (
    overlaps[["END_DAY_1", "END_DAY_2"]].min(axis=1)
    - overlaps[["START_DAY_1", "START_DAY_2"]].max(axis=1)
    + 1
)

In [16]:
# Identify campaign pairs with the longest overlapping periods
overlaps[["CAMPAIGN_1", "CAMPAIGN_2", "OVERLAP_DAYS"]].sort_values("OVERLAP_DAYS", ascending=False).head(10)

,CAMPAIGN_1,CAMPAIGN_2,OVERLAP_DAYS
33,15,20,71
37,15,18,56
30,15,24,50
301,14,15,50
867,27,28,42
836,28,29,40
94,20,23,39
34,15,23,39
621,3,5,35
2,24,25,33


Key findings:

- 30 unique campaigns: 5 TypeA, 19 TypeB, and 6 TypeC.

- Median duration: 48 days (TypeA), 33 days (TypeB), and 65 days (TypeC).

- No missing values or invalid campaign durations.

- 74 campaign pairs overlap in time, with a maximum overlap of 71 days.

- Temporal overlap does not necessarily imply overlapping household participation.

## 1.2 Campaign Participation

In [17]:
campaign_table = pd.read_csv("../data/raw/campaign_table.csv")

In [18]:
campaign_table.head()

,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26
3,TypeA,208,26
4,TypeA,192,26


In [19]:
campaign_table.shape

(7208, 3)

In [20]:
campaign_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 7208 entries, 0 to 7207
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   DESCRIPTION    7208 non-null   str  
 1   household_key  7208 non-null   int64
 2   CAMPAIGN       7208 non-null   int64
dtypes: int64(2), str(1)
memory usage: 169.1 KB


In [21]:
# Check for duplicate household-campaign assignments
campaign_table.duplicated(subset=["household_key", "CAMPAIGN"]).sum()

np.int64(0)

In [22]:
campaign_table["household_key"].nunique()

1584

In [23]:
# Summarize the number of campaigns assigned to each household
campaign_table.groupby("household_key")["CAMPAIGN"].nunique().describe()

count    1584.000000
mean        4.550505
std         2.993988
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max        17.000000
Name: CAMPAIGN, dtype: float64

In [24]:
# Count unique households assigned to each campaign
campaign_sizes = campaign_table.groupby("CAMPAIGN")["household_key"].nunique()

campaign_sizes.describe()

count      30.000000
mean      240.266667
std       304.411579
min        12.000000
25%        69.000000
50%       173.000000
75%       221.500000
max      1133.000000
Name: household_key, dtype: float64

In [25]:
# Identify campaigns with the largest number of households
campaign_sizes.sort_values(ascending=False).head(10)

CAMPAIGN
18    1133
13    1077
8     1076
30     361
26     332
22     276
20     244
14     224
11     214
17     202
Name: household_key, dtype: int64

In [26]:
# Check for campaign IDs missing from the campaign description table
(~campaign_table["CAMPAIGN"].isin(campaign_desc["CAMPAIGN"])).sum()

np.int64(0)

In [27]:
# Create unique pairs of campaigns assigned to the same household
household_pairs = campaign_table.merge(campaign_table, on="household_key",suffixes=("_1", "_2"))
household_pairs = household_pairs[household_pairs["CAMPAIGN_1"] < household_pairs["CAMPAIGN_2"]]

In [28]:
# Identify households assigned to campaigns with overlapping periods
household_overlaps = household_pairs.merge(overlaps[["CAMPAIGN_1", "CAMPAIGN_2", "OVERLAP_DAYS"]], on=["CAMPAIGN_1", "CAMPAIGN_2"], how="inner")

In [29]:
# Count households assigned to overlapping campaigns
household_overlaps["household_key"].nunique()

884

In [30]:
# Count overlapping campaign pairs per household
overlap_counts = household_overlaps.groupby("household_key").size()
overlap_counts.describe()

count    884.000000
mean       3.665158
std        3.307031
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       20.000000
dtype: float64

In [31]:
# Identify households with the most overlapping campaign pairs
overlap_counts.sort_values(ascending=False).head(10)

household_key
2489    20
982     19
2459    19
371     18
2317    17
1453    16
1803    16
676     16
1527    15
2341    15
dtype: int64

**Key findings:**
- 7,208 campaign assignments across 1,584 unique households, with no duplicate household-campaign pairs.
- Households participated in a median of 4 campaigns (maximum: 17).
- Campaign size ranged from 12 to 1,133 households.
- 884 households (55.8%) were assigned to campaigns with overlapping periods.
- Among these households, the median number of overlapping campaign pairs was 2 (maximum: 20)
- Campaign overlap: 884 households (55.8%) were assigned to campaigns with overlapping validity periods. Actual offer receipt dates are unavailable.

## 1.3 Coupons

In [32]:
coupon = pd.read_csv("../data/raw/coupon.csv")

In [33]:
coupon.head()

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
0,10000089061,27160,4
1,10000089064,27754,9
2,10000089073,28897,12
3,51800009050,28919,28
4,52100000076,28929,25


In [34]:
coupon.shape

(124548, 3)

In [35]:
coupon.info()

<class 'pandas.DataFrame'>
RangeIndex: 124548 entries, 0 to 124547
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   COUPON_UPC  124548 non-null  int64
 1   PRODUCT_ID  124548 non-null  int64
 2   CAMPAIGN    124548 non-null  int64
dtypes: int64(3)
memory usage: 2.9 MB


In [36]:
# Check for duplicate campaign-coupon-product combinations
coupon.duplicated(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).sum()

np.int64(5164)

In [37]:
# Count unique campaign-coupon combinations
coupon[["CAMPAIGN", "COUPON_UPC"]].drop_duplicates().shape[0]

1397

In [38]:
# Summarize the number of eligible products per campaign-coupon
products_per_coupon = coupon.groupby(["CAMPAIGN", "COUPON_UPC"])["PRODUCT_ID"].nunique()
products_per_coupon.describe()

count     1397.000000
mean        85.457409
std        688.682458
min          1.000000
25%          6.000000
50%         13.000000
75%         30.000000
max      14477.000000
Name: PRODUCT_ID, dtype: float64

In [39]:
# Inspect duplicate campaign-coupon-product records
coupon[coupon.duplicated(keep=False)].sort_values(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).head(10)

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
52212,50004110076,133534,8
52213,50004110076,133534,8
49849,50004110076,140705,8
49850,50004110076,140705,8
31573,50004110076,849713,8
31574,50004110076,849713,8
52210,50004110076,862863,8
52211,50004110076,862863,8
53123,50004110076,870248,8
53124,50004110076,870248,8


In [40]:
# Count duplicate records by campaign
coupon.loc[coupon.duplicated(), "CAMPAIGN"].value_counts()

CAMPAIGN
27    3391
8      608
30     576
26     292
13     173
18     109
14      13
29       2
Name: count, dtype: int64

In [41]:
# Identify the campaign-coupon combination with the most eligible products
products_per_coupon.sort_values(ascending=False).head(10)

CAMPAIGN  COUPON_UPC 
18        10000085478    14477
13        10000085428    14367
          10000085427     5713
18        10000085477     5686
          10000085479     5474
13        10000085429     5423
8         10000085363     4538
18        10000085475     4294
8         10000085361     4241
13        10000085425     4189
Name: PRODUCT_ID, dtype: int64

In [42]:
# Check the frequency of repeated campaign-coupon-product combinations
coupon.groupby(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).size().value_counts().sort_index()

1     117560
2        688
3        304
4        384
5        173
6         97
7         74
8         18
9         10
10         8
11         7
12        38
13         1
14         7
15         7
16         4
17         2
19         2
Name: count, dtype: int64

In [43]:
# Inspect the highest duplicate frequencies
coupon.groupby(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).size().value_counts().sort_index().tail(8)

11     7
12    38
13     1
14     7
15     7
16     4
17     2
19     2
Name: count, dtype: int64

In [44]:
# Identify the most frequently repeated campaign-coupon-product combinations
coupon.groupby(["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]).size().sort_values(ascending=False).head(10)

CAMPAIGN  COUPON_UPC   PRODUCT_ID
27        57940011080  921912        19
          57940011075  921912        19
          57940018081  6704447       17
          57940018076  6704447       17
          57940018081  7167488       16
          54589314187  1024433       16
          54589314175  1024433       16
          57940018076  7167488       16
          54100027032  1118641       15
          51111122275  5565415       15
dtype: int64

In [45]:
# Check for coupon records referencing unknown campaigns
(~coupon["CAMPAIGN"].isin(campaign_desc["CAMPAIGN"])).sum()

np.int64(0)

In [46]:
# Count distinct coupons per campaign
coupons_per_campaign = coupon.groupby("CAMPAIGN")["COUPON_UPC"].nunique()
coupons_per_campaign.describe()

count     30.000000
mean      46.566667
std       69.301781
min        1.000000
25%       12.250000
50%       17.500000
75%       27.750000
max      209.000000
Name: COUPON_UPC, dtype: float64

In [47]:
# Identify campaigns with the most distinct coupons
coupons_per_campaign.sort_values(ascending=False).head(10)

CAMPAIGN
8     209
18    209
13    207
26    181
30    181
3      34
29     33
28     28
27     27
14     26
Name: COUPON_UPC, dtype: int64

**Key findings:**
- 124,548 records representing 1,397 unique campaign-coupon combinations.
- Campaigns contained 1–209 distinct coupons (median: 17.5).
- Each coupon covered a median of 13 products, with a maximum of 14,477.
- 5,164 excess duplicate records were identified across 8 campaigns, with individual combinations repeated up to 19 times.
- All campaign IDs match `campaign_desc`.
- Duplicate coupon-product mappings require deduplication before analytical joins.

## 1.4 Coupon Redemptions

In [48]:
coupon_redempt = pd.read_csv("../data/raw/coupon_redempt.csv")

In [49]:
coupon_redempt.head()

,household_key,DAY,COUPON_UPC,CAMPAIGN
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8
3,1,597,10000085476,18
4,1,597,54200029176,18


In [50]:
coupon_redempt.shape

(2318, 4)

In [51]:
coupon_redempt.info()

<class 'pandas.DataFrame'>
RangeIndex: 2318 entries, 0 to 2317
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   household_key  2318 non-null   int64
 1   DAY            2318 non-null   int64
 2   COUPON_UPC     2318 non-null   int64
 3   CAMPAIGN       2318 non-null   int64
dtypes: int64(4)
memory usage: 72.6 KB


In [52]:
coupon_redempt.duplicated().sum()

np.int64(0)

In [53]:
coupon_redempt.duplicated(subset=["household_key", "CAMPAIGN", "COUPON_UPC"]).sum()

np.int64(30)

In [54]:
coupon_redempt["household_key"].nunique()

434

In [55]:
# Check whether coupon redemptions match household campaign assignments
redemption_check = coupon_redempt.merge(
    campaign_table[["household_key", "CAMPAIGN"]],
    on=["household_key", "CAMPAIGN"],
    how="left",
    indicator=True
)

redemption_check["_merge"].value_counts()

_merge
both          2318
left_only        0
right_only       0
Name: count, dtype: int64

In [56]:
# Check whether redeemed coupons exist in the coupon reference table
redemption_coupon_check = coupon_redempt.merge(
    coupon[["CAMPAIGN", "COUPON_UPC"]].drop_duplicates(),
    on=["CAMPAIGN", "COUPON_UPC"],
    how="left",
    indicator=True
)

redemption_coupon_check["_merge"].value_counts()

_merge
both          2318
left_only        0
right_only       0
Name: count, dtype: int64

In [57]:
# Add campaign validity periods to coupon redemptions
redemption_dates = coupon_redempt.merge(
    campaign_desc[["CAMPAIGN", "START_DAY", "END_DAY"]],
    on="CAMPAIGN",
    how="left"
)

In [59]:
# Count coupon redemptions outside campaign validity periods
((redemption_dates["DAY"] < redemption_dates["START_DAY"]) | (redemption_dates["DAY"] > redemption_dates["END_DAY"])).sum()

np.int64(0)

In [60]:
# Count coupon redemptions and unique redeeming households by campaign
redemptions_by_campaign = coupon_redempt.groupby("CAMPAIGN").agg(
    redemptions=("COUPON_UPC", "size"),
    households=("household_key", "nunique")
)

redemptions_by_campaign.head()

,redemptions,households
CAMPAIGN,,
1,1,1
2,5,2
3,2,2
4,11,6
5,13,8


In [61]:
# Identify campaigns with the highest number of coupon redemptions
redemptions_by_campaign.sort_values("redemptions", ascending=False).head(10)

,redemptions,households
CAMPAIGN,,
18,653,214
13,629,196
8,372,158
26,73,31
30,64,36
25,61,24
23,60,23
22,47,17
17,45,18


In [62]:
# Identify campaigns with no recorded coupon redemptions
campaign_desc[~campaign_desc["CAMPAIGN"].isin(redemptions_by_campaign.index)]

,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY,DURATION_DAYS


### Household Redemption Rate

In [63]:
# Combine coupon redemptions with campaign participation
campaign_redemption = redemptions_by_campaign.join(campaign_sizes.rename("assigned_households"))
campaign_redemption.head()

,redemptions,households,assigned_households
CAMPAIGN,,,
1,1,1,13
2,5,2,48
3,2,2,12
4,11,6,81
5,13,8,166


In [64]:
# Calculate the percentage of households redeeming at least one coupon
campaign_redemption["redemption_rate_pct"] = (campaign_redemption["households"] / campaign_redemption["assigned_households"] * 100)

campaign_redemption.head()

,redemptions,households,assigned_households,redemption_rate_pct
CAMPAIGN,,,,
1,1,1,13,7.692308
2,5,2,48,4.166667
3,2,2,12,16.666667
4,11,6,81,7.407407
5,13,8,166,4.819277


In [65]:
# Identify campaigns with the highest household redemption rates
campaign_redemption.sort_values("redemption_rate_pct", ascending=False).head(10)

,redemptions,households,assigned_households,redemption_rate_pct
CAMPAIGN,,,,
18,653,214,1133,18.887908
13,629,196,1077,18.198700
3,2,2,12,16.666667
8,372,158,1076,14.684015
25,61,24,187,12.834225
23,60,23,183,12.568306
15,2,2,17,11.764706
19,29,15,130,11.538462
9,43,20,176,11.363636


**Key findings:**
- 2,318 coupon redemptions recorded across 434 unique households.
- No exact duplicate records; 30 repeated household-campaign-coupon combinations occurred on different days.
- All redemptions matched valid household-campaign assignments, existing coupons, and campaign validity periods.
- All 30 campaigns recorded at least one coupon redemption.
- Campaigns 18 (18.89%) and 13 (18.20%) had the highest household redemption rates.
- Redemption rates measure coupon usage, not incremental sales.

## 1.5 Products

In [66]:
product = pd.read_csv("../data/raw/product.csv")

In [67]:
product.head()

,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [68]:
product.shape

(92353, 7)

In [69]:
product.info()

<class 'pandas.DataFrame'>
RangeIndex: 92353 entries, 0 to 92352
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   PRODUCT_ID            92353 non-null  int64
 1   MANUFACTURER          92353 non-null  int64
 2   DEPARTMENT            92353 non-null  str  
 3   BRAND                 92353 non-null  str  
 4   COMMODITY_DESC        92353 non-null  str  
 5   SUB_COMMODITY_DESC    92353 non-null  str  
 6   CURR_SIZE_OF_PRODUCT  92353 non-null  str  
dtypes: int64(2), str(5)
memory usage: 4.9 MB


In [70]:
product.nunique()

PRODUCT_ID              92353
MANUFACTURER             6476
DEPARTMENT                 44
BRAND                       2
COMMODITY_DESC            308
SUB_COMMODITY_DESC       2383
CURR_SIZE_OF_PRODUCT     4345
dtype: int64

In [71]:
product.duplicated().sum()

np.int64(0)

In [73]:
product["DEPARTMENT"].value_counts().head(10)

DEPARTMENT
GROCERY       39021
DRUG GM       31529
PRODUCE        3118
COSMETICS      3011
NUTRITION      2914
MEAT           2544
MEAT-PCKGD     2427
DELI           2354
PASTRY         2149
FLORAL          938
Name: count, dtype: int64

In [74]:
product["BRAND"].value_counts()

BRAND
National    78537
Private     13816
Name: count, dtype: int64

In [75]:
(product["COMMODITY_DESC"] == "NO COMMODITY DESCRIPTION").sum()

np.int64(490)

In [76]:
# Identify departments containing products with unspecified commodity descriptions
product.loc[product["COMMODITY_DESC"] == "NO COMMODITY DESCRIPTION","DEPARTMENT"].value_counts()

DEPARTMENT
MISC. TRANS.    490
Name: count, dtype: int64

In [77]:
# Count unique department-category-subcategory combinations
product[["DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"]].drop_duplicates().shape[0]

3912

In [78]:
# Identify subcategory names used in multiple categories
product.groupby("SUB_COMMODITY_DESC")["COMMODITY_DESC"].nunique().sort_values(ascending=False).head(10)

SUB_COMMODITY_DESC
SEASONAL                         20
MISC                             14
MISC SALES TRANS                 13
DECOR                            12
EASTER BASKETS                   11
SPICES & SEASONINGS              10
EASTER GIFTWARE/DECOR            10
CANDLES                          10
SEASONAL CANDY BAGS-CHOCOLATE     9
BANDANA/SCARVES                   9
Name: COMMODITY_DESC, dtype: int64

In [79]:
# Check for coupon records referencing unknown products
(~coupon["PRODUCT_ID"].isin(product["PRODUCT_ID"])).sum()

np.int64(0)

In [80]:
# Retrieve product attributes for the coupon with the widest coverage
largest_coupon = coupon[(coupon["CAMPAIGN"] == 18) & (coupon["COUPON_UPC"] == 10000085478)]
largest_coupon = largest_coupon.merge(product, on="PRODUCT_ID", how="left")

In [81]:
# Count eligible products by department
largest_coupon.groupby("DEPARTMENT")["PRODUCT_ID"].nunique().sort_values(ascending=False)

DEPARTMENT
DRUG GM            11515
COSMETICS           2920
GROCERY               14
MEAT                  10
NUTRITION              6
MISC. TRANS.           5
RX                     3
COUP/STR & MFG         1
GM MERCH EXP           1
HBC                    1
MISC SALES TRAN        1
Name: PRODUCT_ID, dtype: int64

In [82]:
# Identify the main product categories covered by the largest coupon
largest_coupon.groupby(["DEPARTMENT", "COMMODITY_DESC"])["PRODUCT_ID"].nunique().sort_values(ascending=False).head(10)

DEPARTMENT  COMMODITY_DESC           
COSMETICS   MAKEUP AND TREATMENT         2406
DRUG GM     HAIR CARE PRODUCTS           1724
            ORAL HYGIENE PRODUCTS         907
            HAND/BODY/FACIAL PRODUCTS     675
            VITAMINS                      635
            BABY FOODS                    635
            COLD AND FLU                  593
            DEODORANTS                    516
            SOAP - LIQUID & BAR           490
            ANALGESICS                    485
Name: PRODUCT_ID, dtype: int64

**Key findings:**
- 92,353 unique products across 44 departments, with no duplicate records.
- GROCERY and DRUG GM account for 76.4% of the assortment; 85.0% of products are National brands.
- 490 products have unspecified commodity descriptions, all within MISC. TRANS.
- The product hierarchy contains 3,912 unique department-category-subcategory combinations; subcategory names are not globally unique.
- All coupon product IDs match the product reference table.
- The largest coupon covers 14,477 products, primarily in DRUG GM and COSMETICS.

## 1.6 Household Demographics

In [83]:
hh_demographic = pd.read_csv("../data/raw/hh_demographic.csv")

In [84]:
hh_demographic.head()

,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1
1,Age Group4,X,Level5,Homeowner,Group5,2,None/Unknown,7
2,Age Group2,Y,Level3,Unknown,Group4,3,1,8
3,Age Group2,Y,Level6,Homeowner,Group4,4,2,13
4,Age Group4,Z,Level5,Homeowner,Group3,1,None/Unknown,16


In [85]:
hh_demographic.shape

(801, 8)

In [86]:
hh_demographic.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   classification_1   801 non-null    str  
 1   classification_2   801 non-null    str  
 2   classification_3   801 non-null    str  
 3   HOMEOWNER_DESC     801 non-null    str  
 4   classification_5   801 non-null    str  
 5   classification_4   801 non-null    str  
 6   KID_CATEGORY_DESC  801 non-null    str  
 7   household_key      801 non-null    int64
dtypes: int64(1), str(7)
memory usage: 50.2 KB


In [88]:
hh_demographic.nunique()

classification_1       6
classification_2       3
classification_3      12
HOMEOWNER_DESC         5
classification_5       6
classification_4       5
KID_CATEGORY_DESC      4
household_key        801
dtype: int64

In [89]:
hh_demographic.duplicated().sum()

np.int64(0)

In [92]:
# Count all categories across demographic variables
category_counts = (
    hh_demographic.drop(columns="household_key")
    .melt(var_name="column", value_name="category")
    .groupby(["column", "category"], dropna=False)
    .size()
    .reset_index(name="count")
)

print(category_counts.to_string(index=False))

           column        category  count
   HOMEOWNER_DESC       Homeowner    504
   HOMEOWNER_DESC  Probable Owner     11
   HOMEOWNER_DESC Probable Renter     11
   HOMEOWNER_DESC          Renter     42
   HOMEOWNER_DESC         Unknown    233
KID_CATEGORY_DESC               1    114
KID_CATEGORY_DESC               2     60
KID_CATEGORY_DESC              3+     69
KID_CATEGORY_DESC    None/Unknown    558
 classification_1      Age Group1     46
 classification_1      Age Group2    142
 classification_1      Age Group3    194
 classification_1      Age Group4    288
 classification_1      Age Group5     59
 classification_1      Age Group6     72
 classification_2               X    340
 classification_2               Y    344
 classification_2               Z    117
 classification_3          Level1     61
 classification_3         Level10     11
 classification_3         Level11      5
 classification_3         Level12     11
 classification_3          Level2     74
 classification_

In [94]:
# Check demographic coverage among campaign participants
print(campaign_table["household_key"].drop_duplicates().isin(hh_demographic["household_key"]).value_counts())

household_key
False    824
True     760
Name: count, dtype: int64


In [95]:
# Count campaign assignments per household
household_campaigns = (
    campaign_table.groupby("household_key")
    .size()
    .reset_index(name="campaign_count")
)

# Identify households with available demographic data
household_campaigns["has_demographics"] = (household_campaigns["household_key"].isin(hh_demographic["household_key"]))

In [97]:
# Compare campaign participation by demographic data availability
print(household_campaigns.groupby("has_demographics")["campaign_count"].agg(["count", "mean", "median", "min", "max"]))

                  count      mean  median  min  max
has_demographics                                   
False               824  3.634709     3.0    1   15
True                760  5.543421     5.0    1   17


**Key findings:**
- Demographic data is available for 801 of 2,500 households (32.0%), with no duplicate household IDs or null values.
- Categorical values such as `Unknown` and `None/Unknown` indicate incomplete or ambiguous information.
- Demographic data covers 760 of 1,584 campaign participants (48.0%).
- Households with demographic data received more campaigns on average (5.54 vs. 3.63).
- The observed differences indicate potential selection concerns when restricting analyses to households with demographic information.

## 1.7 Transaction Data

In [98]:
transaction_data = pd.read_csv("../data/raw/transaction_data.csv")

In [99]:
transaction_data.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [100]:
transaction_data.shape

(2595732, 12)

In [101]:
transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2595732 entries, 0 to 2595731
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   household_key      int64  
 1   BASKET_ID          int64  
 2   DAY                int64  
 3   PRODUCT_ID         int64  
 4   QUANTITY           int64  
 5   SALES_VALUE        float64
 6   STORE_ID           int64  
 7   RETAIL_DISC        float64
 8   TRANS_TIME         int64  
 9   WEEK_NO            int64  
 10  COUPON_DISC        float64
 11  COUPON_MATCH_DISC  float64
dtypes: float64(4), int64(8)
memory usage: 237.6 MB


In [102]:
transaction_data.nunique()

household_key          2500
BASKET_ID            276484
DAY                     711
PRODUCT_ID            92339
QUANTITY              11746
SALES_VALUE            5502
STORE_ID                582
RETAIL_DISC            2414
TRANS_TIME             1440
WEEK_NO                 102
COUPON_DISC             415
COUPON_MATCH_DISC        80
dtype: int64

In [104]:
print(transaction_data.isna().sum())

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64


In [105]:
# Check how many households are associated with each basket
print(transaction_data.groupby("BASKET_ID")["household_key"].nunique().value_counts())

household_key
1    276484
Name: count, dtype: int64


In [106]:
# Check how many transaction days are associated with each basket
print(transaction_data.groupby("BASKET_ID")["DAY"].nunique().value_counts())

DAY
1    276484
Name: count, dtype: int64


In [107]:
transaction_data.duplicated().sum()

np.int64(0)

In [108]:
transaction_data.duplicated(subset=["BASKET_ID", "PRODUCT_ID"]).sum()

np.int64(0)

In [109]:
# Count transaction records referencing unknown products
(~transaction_data["PRODUCT_ID"].isin(product["PRODUCT_ID"])).sum()

np.int64(0)

In [111]:
# Check the number of stores associated with each basket
print(transaction_data.groupby("BASKET_ID")["STORE_ID"].nunique().value_counts())

STORE_ID
1    276484
Name: count, dtype: int64


In [112]:
# Check the observation period in days and weeks
print(transaction_data[["DAY", "WEEK_NO"]].agg(["min", "max"]))

     DAY  WEEK_NO
min    1        1
max  711      102


In [113]:
# Check whether each transaction day belongs to a single week
print(transaction_data.groupby("DAY")["WEEK_NO"].nunique().value_counts())

WEEK_NO
1    711
Name: count, dtype: int64


In [115]:
# Examine the distribution of purchased quantities and sales values
print(transaction_data[["QUANTITY", "SALES_VALUE"]].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

                 count        mean          std  min   1%   25%  50%   75%  \
QUANTITY     2595732.0  100.428558  1153.436211  0.0  1.0  1.00  1.0  1.00   
SALES_VALUE  2595732.0    3.104120     4.182274  0.0  0.2  1.29  2.0  3.49   

              99%      max  
QUANTITY     10.0  89638.0  
SALES_VALUE  20.0    840.0  


In [117]:
# Count transactions with zero or negative quantities
print((transaction_data["QUANTITY"] <= 0).sum())

14466


In [118]:
# Count transactions with zero or negative sales values
print((transaction_data["SALES_VALUE"] <= 0).sum())

18850


In [121]:
# Inspect products associated with the highest quantities
print(transaction_data.nlargest(10, "QUANTITY")[["PRODUCT_ID", "QUANTITY", "SALES_VALUE"]].merge(
    product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]],
    on="PRODUCT_ID",
    how="left"
))

   PRODUCT_ID  QUANTITY  SALES_VALUE       DEPARTMENT     COMMODITY_DESC
0     6534178     89638       250.00        KIOSK-GAS  COUPON/MISC ITEMS
1     6544236     85055       210.00  MISC SALES TRAN  COUPON/MISC ITEMS
2     6534178     61335       150.21        KIOSK-GAS  COUPON/MISC ITEMS
3     6534178     51912       110.00        KIOSK-GAS  COUPON/MISC ITEMS
4     6534178     48073       121.10        KIOSK-GAS  COUPON/MISC ITEMS
5     6534178     45475       100.00        KIOSK-GAS  COUPON/MISC ITEMS
6     6534178     41833       115.00        KIOSK-GAS  COUPON/MISC ITEMS
7     6534178     41686       100.00        KIOSK-GAS  COUPON/MISC ITEMS
8     6544236     41485        85.00  MISC SALES TRAN  COUPON/MISC ITEMS
9     6534178     39365        72.00        KIOSK-GAS  COUPON/MISC ITEMS


In [123]:
# Count high-quantity transactions by department
print(transaction_data[transaction_data["QUANTITY"] > 1000].merge(
    product[["PRODUCT_ID", "DEPARTMENT"]],
    on="PRODUCT_ID",
    how="left"
)["DEPARTMENT"].value_counts())

DEPARTMENT
KIOSK-GAS          20237
MISC SALES TRAN     2864
Name: count, dtype: int64


In [125]:
# Compare zero quantities with zero sales values
print(pd.crosstab(transaction_data["QUANTITY"] == 0, transaction_data["SALES_VALUE"] == 0))

SALES_VALUE    False  True 
QUANTITY                   
False        2576815   4451
True              67  14399


In [126]:
# Select transactions with zero quantity or zero sales
zero_transactions = transaction_data[(transaction_data["QUANTITY"] == 0) |(transaction_data["SALES_VALUE"] == 0)].copy()

# Classify combinations of zero quantity and sales
zero_transactions["zero_type"] = "Both zero"

zero_transactions.loc[zero_transactions["QUANTITY"] > 0, "zero_type"] = "Zero sales"

zero_transactions.loc[zero_transactions["SALES_VALUE"] > 0, "zero_type"] = "Zero quantity"

# Count zero-value transactions by department
print(zero_transactions.merge(
    product[["PRODUCT_ID", "DEPARTMENT"]],
    on="PRODUCT_ID",
    how="left"
).groupby(["DEPARTMENT", "zero_type"]).size().unstack(fill_value=0))

zero_type        Both zero  Zero quantity  Zero sales
DEPARTMENT                                           
                      7839              0           0
AUTOMOTIVE               2              0           0
CHEF SHOPPE              2              0           0
CNTRL/STORE SUP          3              0           0
COSMETICS               86              0          64
COUP/STR & MFG           3              0         444
DELI                   162              2          69
DRUG GM               1325             16         662
FLORAL                  77              0           8
FROZEN GROCERY           3              0           0
GARDEN CENTER           25              1          10
GM MERCH EXP             3              0           0
GROCERY               2548             24        1198
MEAT                   165              3         168
MEAT-PCKGD             223              2         938
MISC SALES TRAN         57              0           7
MISC. TRANS.            59  

In [127]:
# Count affected households by zero-value type
print(zero_transactions.groupby("zero_type").agg(
    records=("household_key", "size"),
    households=("household_key", "nunique")
))

               records  households
zero_type                         
Both zero        14399        1986
Zero quantity       67          62
Zero sales        4451        1476


In [128]:
# Identify households with the most zero-value transactions
print(zero_transactions["household_key"].value_counts().head(10))

household_key
1098    124
718     105
979      89
618      78
1795     78
1419     75
2448     73
641      71
936      70
1901     69
Name: count, dtype: int64


In [129]:
# Count households affected by one or more zero-value types
print(zero_transactions.groupby("household_key")["zero_type"].nunique().value_counts())

zero_type
2    1292
1     811
3      43
Name: count, dtype: int64


In [131]:
# Flag transactions with zero quantity or sales value
transaction_data["has_zero"] = ((transaction_data["QUANTITY"] == 0) |(transaction_data["SALES_VALUE"] == 0))

# Identify households assigned to at least one campaign
transaction_data["has_campaign"] = (transaction_data["household_key"].isin(campaign_table["household_key"]))

# Compare zero-value transaction rates by campaign participation
print(transaction_data.groupby("has_campaign")["has_zero"].agg(["count", "sum", "mean"]))

                count    sum      mean
has_campaign                          
False          303816   2085  0.006863
True          2291916  16832  0.007344


In [133]:
# Identify products with blank department labels
blank_department_products = product.loc[product["DEPARTMENT"].str.strip() == "", "PRODUCT_ID"]

# Count zero-value transactions for these products
print(zero_transactions.loc[zero_transactions["PRODUCT_ID"].isin(blank_department_products),    "PRODUCT_ID"].value_counts().head(10))

PRODUCT_ID
5978656    4887
5978648    2867
5126088      16
5993054      13
5993051      11
5977100      10
5126107       8
5978649       7
5978659       5
5978650       3
Name: count, dtype: int64


In [135]:
# Inspect product attributes associated with zero-value records
print(product[product["PRODUCT_ID"].isin([5978656, 5978648])])

       PRODUCT_ID  MANUFACTURER DEPARTMENT     BRAND COMMODITY_DESC  \
55054     5978648             1             National                  
55057     5978656             1             National                  

      SUB_COMMODITY_DESC CURR_SIZE_OF_PRODUCT  
55054                                          
55057                                          


In [137]:
# Check quantity and sales ranges for unidentified products
print(transaction_data.loc[
    transaction_data["PRODUCT_ID"].isin([5978648, 5978656])
].groupby("PRODUCT_ID")[["QUANTITY", "SALES_VALUE"]].agg(
    ["count", "min", "max"]
))

           QUANTITY         SALES_VALUE          
              count min max       count  min  max
PRODUCT_ID                                       
5978648        2867   0   0        2867  0.0  0.0
5978656        4887   0   0        4887  0.0  0.0


In [138]:
# Check transaction values for all products with blank department labels
blank_product_summary = (
    transaction_data[
        transaction_data["PRODUCT_ID"].isin(blank_department_products)
    ]
    .groupby("PRODUCT_ID")
    .agg(
        records=("PRODUCT_ID", "size"),
        max_quantity=("QUANTITY", "max"),
        max_sales=("SALES_VALUE", "max")
    )
    .sort_values("records", ascending=False)
)

print(blank_product_summary.to_string())

            records  max_quantity  max_sales
PRODUCT_ID                                  
5978656        4887             0        0.0
5978648        2867             0        0.0
5126088          16             0        0.0
5993054          13             0        0.0
5993051          11             0        0.0
5977100          10             0        0.0
5126107           8             0        0.0
5978649           7             0        0.0
5978659           5             0        0.0
5126087           3             0        0.0
5126106           3             0        0.0
5978657           3             0        0.0
5978650           3             0        0.0
6693056           2             0        0.0
5993055           1             0        0.0


In [140]:
# Identify baskets containing products with blank department labels
blank_baskets = transaction_data.loc[
    transaction_data["PRODUCT_ID"].isin(blank_department_products),
    "BASKET_ID"
].unique()

# Check whether these baskets have positive total sales
print(transaction_data.loc[
    transaction_data["BASKET_ID"].isin(blank_baskets)
].groupby("BASKET_ID")["SALES_VALUE"].sum().eq(0).value_counts())

SALES_VALUE
False    7075
True      317
Name: count, dtype: int64


In [143]:
# Count zero-value records by product hierarchy and anomaly type
zero_subcategories = (
    zero_transactions.merge(
        product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"]],
        on="PRODUCT_ID",
        how="left"
    )
    .groupby(
        ["DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC", "zero_type"]
    )
    .size()
    .unstack(fill_value=0)
)

# Rank product subcategories by the total number of zero-value records
zero_subcategories["total"] = zero_subcategories.sum(axis=1)

print(zero_subcategories.sort_values("total", ascending=False).head(30))

zero_type                                                                     Both zero  \
DEPARTMENT     COMMODITY_DESC                 SUB_COMMODITY_DESC                          
                                                                                   7839   
COUP/STR & MFG COUPONS/STORE & MFG            COUPONS/STORE & MFG                     3   
PASTRY         COOKIES                        COOKIES: REGULAR                        8   
GROCERY        BAG SNACKS                     PRETZELS                               10   
DRUG GM        CANDY - CHECKLANE              CANDY BARS (SINGLES)(INCLUDING        112   
GROCERY        ICE CREAM/MILK/SHERBTS         PREMIUM                                16   
MEAT-PCKGD     DINNER SAUSAGE                 SMOKED/COOKED                          15   
               LUNCHMEAT                      POULTRY                                 6   
               HOT DOGS                       PREMIUM - MEAT                          7   

In [145]:
# Count all transaction records by product hierarchy
subcategory_totals = (
    transaction_data.merge(
        product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"]],
        on="PRODUCT_ID",
        how="left"
    )
    .groupby(["DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"])
    .size()
    .rename("all_records")
)

# Calculate the percentage of zero-value records in each subcategory
subcategory_rates = zero_subcategories.join(subcategory_totals)

subcategory_rates["zero_pct"] = (
    subcategory_rates["total"] / subcategory_rates["all_records"] * 100
).round(2)

# Show subcategories with at least 100 records, excluding blank departments
print(subcategory_rates.reset_index().loc[
    lambda df: (df["all_records"] >= 100)
    & (df["DEPARTMENT"].str.strip() != "")
].sort_values("zero_pct", ascending=False).head(20))

           DEPARTMENT          COMMODITY_DESC             SUB_COMMODITY_DESC  \
35     COUP/STR & MFG     COUPONS/STORE & MFG            COUPONS/STORE & MFG   
1177           PASTRY                 COOKIES               COOKIES: REGULAR   
138           DRUG GM     CANDLES/ACCESSORIES               CANDLES SEASONAL   
755           GROCERY     FROZEN PIE/DESSERTS  FROZEN FRUIT PIES  & COBBLERS   
104           DRUG GM    AUDIO/VIDEO PRODUCTS                      SPHE 9.99   
541     GARDEN CENTER           GARDEN CENTER     HANGING BASKETS (BLOOMING)   
1341       RESTAURANT             COFFEE SHOP         SV BEV: COFFEE DINE IN   
24          COSMETICS    MAKEUP AND TREATMENT               LOREAL COSMETICS   
822           GROCERY  HOUSEHOLD CLEANG NEEDS        SALT/WATER CONDITIONERS   
58               DELI           DELI SUPPLIES               MISC SALES TRANS   
754           GROCERY     FROZEN PIE/DESSERTS              FROZEN CREAM PIES   
680           GROCERY       COUPON/MISC 

In [148]:
# Examine the distribution of transaction discounts
print(transaction_data[["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]].describe(percentiles=[0.01, 0.5, 0.99]))

        RETAIL_DISC   COUPON_DISC  COUPON_MATCH_DISC
count  2.595732e+06  2.595732e+06       2.595732e+06
mean  -5.387054e-01 -1.641600e-02      -2.918564e-03
std    1.249191e+00  2.168410e-01       3.969004e-02
min   -1.800000e+02 -5.593000e+01      -7.700000e+00
1%    -4.990000e+00 -5.000000e-01       0.000000e+00
50%   -1.000000e-02  0.000000e+00       0.000000e+00
99%    0.000000e+00  0.000000e+00       0.000000e+00
max    3.990000e+00  0.000000e+00       0.000000e+00


In [149]:
# Select zero-sales transactions with positive quantities
zero_sales = transaction_data.loc[
    (transaction_data["QUANTITY"] > 0) &
    (transaction_data["SALES_VALUE"] == 0),
    ["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]
]

# Count positive, negative and zero discounts
print(pd.DataFrame({
    "negative": (zero_sales < 0).sum(),
    "zero": (zero_sales == 0).sum(),
    "positive": (zero_sales > 0).sum()
}))

                   negative  zero  positive
RETAIL_DISC            3748   703         0
COUPON_DISC              43  4408         0
COUPON_MATCH_DISC        12  4439         0


In [150]:
# Compare retail discount rates between zero-sales and positive-sales records
positive_quantity = transaction_data[transaction_data["QUANTITY"] > 0]

print(pd.crosstab(
    positive_quantity["SALES_VALUE"] == 0,
    positive_quantity["RETAIL_DISC"] < 0,
    normalize="index"
).mul(100).round(2))

RETAIL_DISC  False  True 
SALES_VALUE              
False        49.58  50.42
True         15.79  84.21


In [152]:
# Select records with positive quantity and zero sales
zero_sales_records = transaction_data[
    (transaction_data["QUANTITY"] > 0) &
    (transaction_data["SALES_VALUE"] == 0)
]

# Check whether all three discount fields equal zero
print(zero_sales_records[["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]].eq(0).all(axis=1).value_counts())

False    3749
True      702
Name: count, dtype: int64


In [153]:
# Select zero-sales records without recorded discounts
no_discount = zero_sales_records[
    zero_sales_records[
        ["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]
    ].eq(0).all(axis=1)
]

# Identify product categories with the most unexplained zero-sales records
print((
    no_discount.merge(
        product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"]],
        on="PRODUCT_ID",
        how="left"
    )
    .value_counts(["DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"])
    .head(15)
))

DEPARTMENT       COMMODITY_DESC       SUB_COMMODITY_DESC     
COUP/STR & MFG   COUPONS/STORE & MFG  COUPONS/STORE & MFG        444
GROCERY          BAG SNACKS           PRETZELS                   184
                                      BAGGED CHEESE SNACKS        17
                                      TORTILLA/NACHO CHIPS        10
                                      CORN CHIPS                   7
                                      POTATO CHIPS                 4
MISC SALES TRAN  COUPON/MISC ITEMS    MISC SALES TRANS             4
PASTRY           ROLLS                ROLLS: DINNER                4
GROCERY          BAG SNACKS           MISC BAG SNACKS              3
PRODUCE          TROPICAL FRUIT       BANANAS                      2
GROCERY          COUPON/MISC ITEMS    MISC SALES TRANS             2
                 BAKED SWEET GOODS    SWEET GOODS - FULL SIZE      2
MISC SALES TRAN  COUPON/MISC ITEMS    PRODUCE DEPT KEY RING        2
                                      MEA

In [154]:
# Identify products responsible for zero sales without recorded discounts
print((
    no_discount.groupby("PRODUCT_ID")
    .agg(
        records=("PRODUCT_ID", "size"),
        min_quantity=("QUANTITY", "min"),
        max_quantity=("QUANTITY", "max")
    )
    .merge(
        product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC"]],
        on="PRODUCT_ID",
        how="left"
    )
    .sort_values("records", ascending=False)
    .head(15)
))

    PRODUCT_ID  records  min_quantity  max_quantity      DEPARTMENT  \
38     1053460      323             1             3  COUP/STR & MFG   
7       897752       54             1             1         GROCERY   
43     1079228       46             1             1         GROCERY   
30     1013587       42             1             1  COUP/STR & MFG   
24      991268       38             1             2         GROCERY   
52     5574954       29             1             2  COUP/STR & MFG   
5       885430       20             1             1         GROCERY   
0       744587       19             1             1  COUP/STR & MFG   
35     1049131       16             1             2         GROCERY   
47     1105074       11             1             1  COUP/STR & MFG   
1       837963       11             1             1         GROCERY   
4       876660        6             1             1         GROCERY   
42     1077422        6             1             2         GROCERY   
31    

In [155]:
# Select the ten products most frequently recorded with zero sales and no discounts
top_product_ids = no_discount["PRODUCT_ID"].value_counts().head(10).index

# Retrieve all transactions for these products
top_product_transactions = transaction_data[
    transaction_data["PRODUCT_ID"].isin(top_product_ids)
]

# Compare zero-sales and positive-sales records for each product
print(pd.crosstab(
    top_product_transactions["PRODUCT_ID"],
    top_product_transactions["SALES_VALUE"] == 0
))

SALES_VALUE  False  True 
PRODUCT_ID               
744587           0     19
837963          29     17
885430          73     32
897752         307     93
991268         192     54
1013587          0     42
1049131         66     30
1053460          0    323
1079228        273     93
5574954          0     29


In [156]:
# Calculate the frequency of nonzero discounts
discount_columns = ["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]

print((transaction_data[discount_columns].ne(0).mean() * 100).round(2))

RETAIL_DISC          50.20
COUPON_DISC           1.40
COUPON_MATCH_DISC     0.67
dtype: float64


In [157]:
# Select transactions with positive retail discount values
positive_retail = transaction_data[transaction_data["RETAIL_DISC"] > 0]

# Count positive retail discount records
positive_retail.shape[0]

36

In [158]:
# Identify product categories associated with positive retail discounts
print((
    positive_retail.merge(
        product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]],
        on="PRODUCT_ID",
        how="left"
    )
    .value_counts(["DEPARTMENT", "COMMODITY_DESC"])
    .head(10)
))

DEPARTMENT  COMMODITY_DESC        
DRUG GM     CANDY - CHECKLANE         7
GROCERY     CAT FOOD                  3
            SOFT DRINKS               2
            MEAT - SHELF STABLE       1
            FLUID MILK PRODUCTS       1
            FRZN VEGETABLE/VEG DSH    1
MEAT-PCKGD  BACON                     1
GROCERY     MILK BY-PRODUCTS          1
MEAT-PCKGD  HOT DOGS                  1
            HEAT/SERVE                1
Name: count, dtype: int64


In [159]:
# Calculate total sales value for each basket
basket_sales = transaction_data.groupby("BASKET_ID")["SALES_VALUE"].sum()

# Examine the distribution of basket values
print(basket_sales.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]))

count    276484.000000
mean         29.142602
std          36.101301
min           0.000000
1%            0.590000
25%           6.960000
50%          17.070000
75%          36.280000
99%         174.123400
max         961.490000
Name: SALES_VALUE, dtype: float64


In [160]:
# Count baskets with zero total sales
(basket_sales == 0).sum()

np.int64(942)

In [161]:
# Identify baskets with zero total sales
zero_basket_ids = basket_sales[basket_sales == 0].index

# Check whether zero-sales baskets contain positive quantities
print((
    transaction_data[transaction_data["BASKET_ID"].isin(zero_basket_ids)]
    .groupby("BASKET_ID")["QUANTITY"]
    .max()
    .gt(0)
    .value_counts()
))

QUANTITY
False    592
True     350
Name: count, dtype: int64


**Key findings:**
- The dataset contains 2,595,732 transaction lines across 276,484 baskets, 2,500 households, and 582 stores, covering days 1–711.
- No null values, exact duplicate rows, duplicate basket-product combinations, or unmatched product IDs were found. Each basket belongs to one household, day, and store.
- Basket sales are right-skewed (mean: $29.14; median: $17.07).
- Extreme `QUANTITY` values are concentrated in `KIOSK-GAS` and `MISC SALES TRAN`, indicating that quantities should not be aggregated indiscriminately across departments.
- Zero-value records include 7,839 entries for 15 products with blank descriptions. These products always have zero quantity and sales.
- Retail discounts occur in 50.20% of transaction lines, compared with 1.40% for manufacturer coupons and 0.67% for coupon matching.
- Of 942 zero-sales baskets, 592 contain no positive quantities, while 350 contain at least one positive-quantity item.
- Zero-value records require differentiated treatment when defining sales and purchase-frequency KPIs.

## 1.8 Causal Data

In [162]:
causal_data = pd.read_csv("../data/raw/causal_data.csv")

In [163]:
causal_data.head()

,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A


In [164]:
causal_data.shape

(36786524, 5)

In [165]:
causal_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 36786524 entries, 0 to 36786523
Data columns (total 5 columns):
 #   Column      Dtype
---  ------      -----
 0   PRODUCT_ID  int64
 1   STORE_ID    int64
 2   WEEK_NO     int64
 3   display     str  
 4   mailer      str  
dtypes: int64(3), str(2)
memory usage: 1.4 GB


In [166]:
causal_data.nunique()

PRODUCT_ID    68377
STORE_ID        115
WEEK_NO          93
display          10
mailer           11
dtype: int64

In [167]:
# Count records by display type
causal_data["display"].value_counts(dropna=False)

display
0    21038745
9     2699467
5     2575289
7     2362118
3     2073738
6     1816021
2     1812840
1     1102141
A      713180
4      592985
Name: count, dtype: int64

In [168]:
# Count records by mailer type
causal_data["mailer"].value_counts(dropna=False)

mailer
A    17106789
0    11534183
D     4467453
H     1560395
F     1077549
J      306924
L      301327
C      291059
X      120823
Z       19453
P         569
Name: count, dtype: int64

In [171]:
# Select positive-quantity items from zero-sales baskets
zero_positive_lines = transaction_data.loc[
    transaction_data["BASKET_ID"].isin(zero_basket_ids),
    ["BASKET_ID", "PRODUCT_ID", "STORE_ID", "WEEK_NO"]
]

# Select products advertised as free in mailers
free_promotions = causal_data.loc[
    causal_data["mailer"].isin(["X", "Z"]),
    ["PRODUCT_ID", "STORE_ID", "WEEK_NO", "mailer"]
]

# Match purchased products to free-product promotions
free_matches = zero_positive_lines.merge(
    free_promotions,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="inner"
)

# Count matching baskets
free_matches["BASKET_ID"].nunique()

0

In [173]:
# Select positive-quantity items from zero-sales baskets
zero_positive_lines = transaction_data.loc[
    transaction_data["BASKET_ID"].isin(zero_basket_ids)
    & (transaction_data["QUANTITY"] > 0),
    ["BASKET_ID", "PRODUCT_ID", "STORE_ID", "WEEK_NO"]
]

# Select matching product-store-week combinations
causal_keys = causal_data.loc[
    causal_data["PRODUCT_ID"].isin(zero_positive_lines["PRODUCT_ID"]),
    ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
].drop_duplicates()

# Check promotion data coverage
coverage = zero_positive_lines.merge(
    causal_keys,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="left",
    indicator=True
)

print(coverage["_merge"].value_counts())

_merge
left_only     349
both            2
right_only      0
Name: count, dtype: int64


In [174]:
# Count duplicate product-store-week combinations
causal_data.duplicated(subset=["PRODUCT_ID", "STORE_ID", "WEEK_NO"]).sum()

np.int64(15245)

In [175]:
# Count exact duplicate records across all five columns
causal_data.duplicated().sum()

np.int64(0)

In [176]:
# Inspect product-store-week combinations appearing multiple times
causal_data[
    causal_data.duplicated(
        subset=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
        keep=False
    )
].sort_values(
    ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
).head(20)

,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
45594,27658,361,59,0,D
45595,27658,361,59,6,D
45604,27658,361,76,0,D
45605,27658,361,76,6,D
49368,27658,34280,89,0,D
49369,27658,34280,89,6,D
49371,27658,34280,91,0,D
49372,27658,34280,91,6,D
49373,27658,34280,92,0,D
49374,27658,34280,92,6,D


In [178]:
# Select all records with repeated product-store-week keys
repeated_promotions = causal_data[
    causal_data.duplicated(
        subset=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
        keep=False
    )
]

# Count distinct display and mailer codes within repeated keys
promotion_variations = (
    repeated_promotions
    .groupby(["PRODUCT_ID", "STORE_ID", "WEEK_NO"])
    .agg(
        display_types=("display", "nunique"),
        mailer_types=("mailer", "nunique")
    )
)

# Summarize the types of variation
print(promotion_variations.value_counts())

display_types  mailer_types
2              1               15140
               2                  68
1              2                  37
Name: count, dtype: int64


In [179]:
# Identify display combinations within repeated product-store-week keys
display_combinations = (
    repeated_promotions
    .groupby(["PRODUCT_ID", "STORE_ID", "WEEK_NO"])
    .agg(
        mailer=("mailer", "first"),
        mailer_types=("mailer", "nunique"),
        displays=("display", lambda x: " + ".join(sorted(x.unique())))
    )
)

# Count display combinations where the mailer code is unchanged
print(display_combinations.loc[
    display_combinations["mailer_types"] == 1
].value_counts(["mailer", "displays"]).head(30))

mailer  displays
A       0 + 1       3850
        0 + 7       3344
D       0 + 7       1558
A       0 + 9        766
        0 + 5        647
        0 + 3        645
        0 + 2        592
D       0 + 6        471
A       0 + 6        422
D       0 + 1        421
H       0 + 7        284
        0 + 1        263
D       0 + 3        260
A       0 + 4        253
D       0 + 2        249
        0 + 9        247
H       0 + 5        108
A       0 + A        107
D       0 + 5         85
H       0 + 9         58
J       0 + 7         51
D       0 + 4         47
H       0 + 6         46
        0 + A         43
        0 + 3         42
        0 + 2         31
        0 + 4         30
J       0 + 9         30
L       0 + 6         29
X       0 + 3         25
Name: count, dtype: int64


In [180]:
# Check whether every repeated key with unchanged mailer includes display code 0
display_combinations.loc[
    display_combinations["mailer_types"] == 1,
    "displays"
].str.startswith("0 + ").value_counts()

displays
True    15140
Name: count, dtype: int64

In [181]:
causal_data.isna().sum()

PRODUCT_ID    0
STORE_ID      0
WEEK_NO       0
display       0
mailer        0
dtype: int64

In [182]:
causal_data["WEEK_NO"].agg(["min", "max"])

min      9
max    101
Name: WEEK_NO, dtype: int64

In [183]:
# Check product IDs against the product catalog
missing_products = ~pd.Series(causal_data["PRODUCT_ID"].unique()).isin(product["PRODUCT_ID"])

# Check store IDs against transaction history
missing_stores = ~pd.Series(causal_data["STORE_ID"].unique()).isin(transaction_data["STORE_ID"])

# Display unmatched identifier counts
print("Unmatched product IDs:", missing_products.sum())
print("Unmatched store IDs:", missing_stores.sum())

Unmatched product IDs: 0
Unmatched store IDs: 0


In [184]:
# Select unique product-store-week combinations from promotion data
causal_keys = causal_data[["PRODUCT_ID", "STORE_ID", "WEEK_NO"]].drop_duplicates()

# Match transaction lines to promotion data
transaction_coverage = transaction_data[["PRODUCT_ID", "STORE_ID", "WEEK_NO"]].merge(
    causal_keys,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="left",
    indicator=True
)

# Calculate the percentage of matched and unmatched transaction lines
print((transaction_coverage["_merge"].value_counts(normalize=True) * 100).round(2))

_merge
left_only     78.28
both          21.72
right_only     0.00
Name: proportion, dtype: float64


In [186]:
# Compare transaction coverage by store and week availability
coverage_by_scope = (
    transaction_coverage.assign(
        in_causal_store=transaction_coverage["STORE_ID"].isin(
            causal_data["STORE_ID"].unique()
        ),
        in_causal_week=transaction_coverage["WEEK_NO"].between(9, 101),
        matched=transaction_coverage["_merge"].eq("both")
    )
    .groupby(["in_causal_store", "in_causal_week"])["matched"]
    .agg(["count", "sum", "mean"])
)

print(coverage_by_scope)

                                  count     sum      mean
in_causal_store in_causal_week                           
False           False              1667       0  0.000000
                True              35455       0  0.000000
True            False             71966       0  0.000000
                True            2486644  563774  0.226721


In [187]:
# Count combinations of in-store display and mailer exposure
print(pd.crosstab(
    causal_data["display"].ne("0"),
    causal_data["mailer"].ne("0"),
    rownames=["has_display"],
    colnames=["has_mailer"],
    margins=True
))

has_mailer      False      True       All
has_display                              
False               0  21038745  21038745
True         11534183   4213596  15747779
All          11534183  25252341  36786524


In [189]:
# Identify store-week combinations covered by promotion data
covered_store_weeks = causal_data[
    ["STORE_ID", "WEEK_NO"]
].drop_duplicates()

# Check store-week coverage for unmatched transaction lines
print((
    transaction_coverage.loc[
        transaction_coverage["_merge"] == "left_only",
        ["STORE_ID", "WEEK_NO"]
    ]
    .merge(
        covered_store_weeks,
        on=["STORE_ID", "WEEK_NO"],
        how="left",
        indicator=True
    )["_merge"]
    .value_counts()
))

_merge
both          1922783
left_only      109175
right_only          0
Name: count, dtype: int64


**Key findings:**
- The dataset contains 36,786,524 records covering 68,377 products, 115 stores, and weeks 9–101, with no null values.
- There are 15,245 repeated product–store–week combinations but no exact duplicate rows.
- In 99.31% of repeated combinations, the mailer code is unchanged while display codes include `0` and one nonzero value. This is consistent with possible within-week exposure changes, which cannot be verified without daily data.
- Every record indicates at least one form of display or mailer exposure, suggesting that unexposed product–store–week combinations are omitted.
- Of all transaction lines, 21.72% have a matching exposure record, 74.07% have no product-level match within a covered store-week, and 4.21% fall outside observed store-week combinations.
- Missing product-level matches within covered store-weeks may indicate no recorded exposure, but this interpretation requires an explicit modeling assumption.
- Repeated exposure records must be consolidated before joining with transactions to avoid duplicating sales.